[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/junlinghu/book-agents/blob/main/tutorial/5-memory.ipynb)

## Colab setup

Run the next cell first on Google Colab. It sparse-checkouts this public repo and installs packages only when the kernel is Colab. Local Jupyter and VS Code skip that work.

An API key is required. Set `OPENAI_API_KEY` in Colab secrets (the key icon, secret name `OPENAI_API_KEY`) or in an environment cell: `os.environ["OPENAI_API_KEY"] = "sk-..."`. Do that before the lesson setup cell. Locally, put the same name in the repository-root `.env`. A missing key stops the notebook. The key is not printed.

Edits you make in Colab stay in that session. They do not push to GitHub.

In [ ]:
# Colab setup. Run this cell before the other code cells.
# Local Jupyter and VS Code skip the clone and the install.
#
# An API key is required. On Google Colab, set OPENAI_API_KEY in one of these ways:
#   * Secrets (the key icon): a secret named OPENAI_API_KEY
#   * an environment cell: os.environ["OPENAI_API_KEY"] = "sk-..."
# Locally, put the key in the repository-root .env.
# Do not commit a key. This cell does not print the value.
# Edits you make in Colab stay in the session. They do not push to GitHub.

import os
import sys
from pathlib import Path


def in_colab():
    """True on Google Colab. False in local Jupyter and VS Code."""
    try:
        import google.colab
    except ImportError:
        return False
    return Path("/content").is_dir() and google.colab is not None


if not in_colab():
    print("Not Colab. Skipped clone and pip install.")
else:
    import shutil
    import subprocess

    missing = []
    for module_name, requirement in (
        ("openai", "openai>=1.40.0"),
        ("dotenv", "python-dotenv>=1.0.1"),
        ("httpx", "httpx>=0.27.0"),
    ):
        try:
            __import__(module_name)
        except ImportError:
            missing.append(requirement)
    if missing:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *missing])

    repo = Path("/content/book-agents")
    marker = repo / "tutorial" / "common" / "client.py"
    if not marker.is_file():
        if repo.exists() and not (repo / ".git").exists():
            raise RuntimeError(
                str(repo) + " exists but is not a git checkout. "
                "Move that folder aside and run this cell again."
            )
        if repo.exists():
            shutil.rmtree(repo)
        url = "https://github.com/junlinghu/book-agents.git"
        try:
            subprocess.check_call([
                "git", "clone", "--depth", "1", "--filter=blob:none", "--sparse",
                url, str(repo),
            ])
            subprocess.check_call([
                "git", "-C", str(repo), "sparse-checkout", "set", "tutorial",
            ])
        except subprocess.CalledProcessError:
            if repo.exists():
                shutil.rmtree(repo)
            subprocess.check_call(["git", "clone", "--depth", "1", url, str(repo)])
        if not marker.is_file():
            raise RuntimeError(
                "Colab setup could not find tutorial/common/client.py after cloning."
            )

    os.chdir(repo)
    if str(repo) not in sys.path:
        sys.path.insert(0, str(repo))

    if not os.environ.get("OPENAI_API_KEY", "").strip():
        try:
            from google.colab import userdata
            secret = userdata.get("OPENAI_API_KEY")
        except Exception:
            secret = ""
        if secret and str(secret).strip():
            os.environ["OPENAI_API_KEY"] = str(secret).strip()

    if not os.environ.get("OPENAI_API_KEY", "").strip():
        raise RuntimeError(
            "OPENAI_API_KEY is empty. On Colab, add a secret named OPENAI_API_KEY "
            "(the key icon) and rerun this cell. Locally, copy .env.example to .env "
            "and paste a key from https://platform.openai.com/api-keys. Never commit .env."
        )
    print("Colab: tutorial/ is ready. OPENAI_API_KEY is set (value hidden).")


# 5. Session and durable memory

The message list is the session. Close it and the sentences are gone. A guest allergy has to outlive that.

This notebook adds `memory_set` and `memory_search`. They read and write `tutorial/var/memory.json`. Turn 1 saves a constraint. Turn 2 builds a new message list on purpose, then searches the file. The loop is imported.

Shared helpers this lesson needs are imported, not copied from earlier notebooks. You can run this file by itself.

## Setup

This notebook calls the OpenAI Chat Completions API. `OPENAI_API_KEY` is required. Locally the key is loaded from the repository-root `.env` by `tutorial/common/client.py`. On Colab, the setup cell above reads a secret of the same name. The value is not printed. A missing key stops the run.

Companion notes: `5-memory.md`.

In [ ]:
import sys
from pathlib import Path


def find_root():
    """Repo root, whether the kernel started here or in tutorial."""
    here = Path.cwd().resolve()
    for candidate in [here, *here.parents]:
        marker = candidate / "tutorial" / "common" / "client.py"
        if marker.is_file():
            return candidate
    raise RuntimeError(
        "Open this notebook inside the book-agents repository. "
        "The setup cell looks for tutorial/common/client.py."
    )


ROOT = find_root()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from tutorial.common.client import describe_runtime
from tutorial.runtime import chat, require_key

LESSON = '5-memory'
LESSON_NUMBER = 5
require_key()
print("tutorial:", LESSON)
print(describe_runtime())


## Shared pieces

These imports are the earlier pieces this lesson uses. They come from `tutorial.common` and `tutorial.runtime`. You do not need to run the previous notebook first.


In [ ]:
from tutorial.common.harness import check, system_text
from tutorial.common.loop import run_agent


## Tutorial 5 — durable memory

Rows live in JSON. A new message list does not delete the file.

In [ ]:
import json

from tutorial.common.harness import VAR, register

MEMORY_PATH = VAR / "memory.json"


def reset_memory():
    """Start this lesson from an empty durable store. Session messages are separate."""
    MEMORY_PATH.parent.mkdir(parents=True, exist_ok=True)
    MEMORY_PATH.write_text("[]\n", encoding="utf-8")


def _load_memory():
    if not MEMORY_PATH.is_file():
        return []
    return json.loads(MEMORY_PATH.read_text(encoding="utf-8"))


def _save_memory(rows):
    MEMORY_PATH.parent.mkdir(parents=True, exist_ok=True)
    MEMORY_PATH.write_text(json.dumps(rows, indent=2) + "\n", encoding="utf-8")


def memory_set(args):
    text = str(args.get("text", "")).strip()
    scope = str(args.get("scope", "")).strip()
    kind = str(args.get("kind", "")).strip()
    source = str(args.get("source", "")).strip()
    if kind not in {"preference", "constraint", "belief"}:
        return "ERROR: kind must be preference, constraint, or belief."
    if not text or not scope or not source:
        return "ERROR: text, scope, and source are required."
    if len(text) > 500:
        return "ERROR: text is too long."
    rows = _load_memory()
    record = {
        "id": "mem_" + str(len(rows) + 1),
        "text": text,
        "scope": scope,
        "kind": kind,
        "source": source,
        "forgotten": False,
    }
    rows.append(record)
    _save_memory(rows)
    return "SAVED id=" + record["id"] + " kind=" + kind + " scope=" + scope


def memory_search(args):
    query = str(args.get("query", "")).strip().lower()
    scope = str(args.get("scope", "")).strip()
    rows = [row for row in _load_memory() if not row.get("forgotten")]
    if scope:
        rows = [row for row in rows if row["scope"] == scope]
    if query:
        rows = [
            row for row in rows
            if query in row["text"].lower() or query in row["scope"].lower()
        ]
    if not rows:
        return "MATCHES\n(none)"
    lines = ["MATCHES"]
    for row in rows:
        lines.append(
            "- id=" + row["id"]
            + " kind=" + row["kind"]
            + " scope=" + row["scope"]
            + " text=" + row["text"]
        )
    return "\n".join(lines)


register(
    "memory_set",
    "Append one durable memory row to JSON. scope is customer:<name> or shop. "
    "kind is preference, constraint, or belief. This survives a new message list.",
    {
        "text": {"type": "string"},
        "scope": {"type": "string"},
        "kind": {"type": "string"},
        "source": {"type": "string"},
    },
    ["text", "scope", "kind", "source"],
    memory_set,
)
register(
    "memory_search",
    "Search durable memory. Session chat is not searched. Forgotten rows are skipped.",
    {
        "query": {"type": "string"},
        "scope": {"type": "string", "description": "Optional exact scope, such as customer:priya or shop."},
    },
    [],
    memory_search,
)


## Run

`system_text()` mentions only the tools imported above. Run this cell after the ones above. Each model turn calls the Chat Completions API. A check prints a warning when the wording differs from the expected trace, and the notebook continues.

In [ ]:
reset_memory()
print("memory file before turn 1:", MEMORY_PATH)
print(MEMORY_PATH.read_text(encoding="utf-8"))

print("=== turn 1: save during this session ===")
saved = run_agent(
    "Remember that Priya is allergic to almonds.",
    system_text(),
)
print("ANSWER:", saved["text"])
stored = MEMORY_PATH.read_text(encoding="utf-8")
print("memory file after turn 1:")
print(stored)
check(saved["stopped"] == "final", "save turn finished")
check("almond" in stored.lower(), "the JSON file holds the constraint")

print("=== turn 2: new message list, same file ===")
recalled = run_agent(
    "What constraint do we have for Priya before I recommend a pastry?",
    system_text(),
)
print("ANSWER:", recalled["text"])
check(recalled["stopped"] == "final", "recall turn finished")
check("memory_search" in "\n".join(recalled["tool_log"]), "recall used the memory tool")
check("almond" in recalled["text"].lower(), "the new session found the durable constraint")
